# Referência Sequencial: Filtro Gaussiano Iterativo 2D

Este notebook serve como referência para a implementação sequencial em Python do Filtro Gaussiano Iterativo 2D, a ser comparada com a versão paralela em C na disciplina DEC107 - Processamento Paralelo.

## O que é o Filtro Gaussiano?

O filtro gaussiano é uma técnica de processamento de imagens que suaviza ou desfoca a imagem, reduzindo o ruído e detalhes finos. Ele funciona aplicando uma média ponderada dos pixels vizinhos, onde os pesos são maiores para os pixels mais próximos ao centro, seguindo uma distribuição gaussiana.

## Como funciona a convolução com máscara?

A convolução usa um kernel (ou máscara), como uma matriz 3x3 ou 5x5, que é deslizada sobre a imagem. Para cada posição, multiplica-se elemento a elemento o kernel com a região correspondente da imagem e soma-se o resultado. O efeito iterativo aumenta o desfoque a cada aplicação.

In [ ]:
import numpy as np

def create_gaussian_kernel(size, sigma=1.0):
    # Valida se o tamanho é ímpar
    if size % 2 == 0:
        raise ValueError("O tamanho do kernel deve ser ímpar.")
    
    # Cria coordenadas centradas
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    
    # Calcula o kernel gaussiano 2D
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    
    # Normaliza para que a soma seja 1
    kernel /= kernel.sum()
    
    return kernel

In [ ]:
def apply_convolution(image, kernel):
    # Obtém dimensões
    img_height, img_width = image.shape
    k_height, k_width = kernel.shape
    
    # Calcula padding necessário
    pad_h = k_height // 2
    pad_w = k_width // 2
    
    # Aplica padding por replicação de bordas
    padded_image = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    
    # Inicializa imagem de saída
    output = np.zeros_like(image)
    
    # Aplica convolução
    for i in range(img_height):
        for j in range(img_width):
            # Extrai região
            region = padded_image[i:i+k_height, j:j+k_width]
            # Multiplica elemento a elemento e soma
            output[i, j] = np.sum(region * kernel)
    
    return output

In [ ]:
def iterative_gaussian_blur(image, kernel_size, iterations, sigma=1.0):
    # Cria o kernel uma vez
    kernel = create_gaussian_kernel(kernel_size, sigma)
    
    # Aplica a convolução iterativamente
    current_image = image.copy()
    for _ in range(iterations):
        current_image = apply_convolution(current_image, kernel)
    
    return current_image

## Exemplo de teste simples

Usaremos uma matriz 2D sintética para testar o filtro. Esta implementação em Python serve como referência para validar a versão em C.

In [ ]:
# Cria uma matriz 5x5 simples
image = np.array([
    [1, 2, 3, 2, 1],
    [2, 4, 6, 4, 2],
    [3, 6, 9, 6, 3],
    [2, 4, 6, 4, 2],
    [1, 2, 3, 2, 1]
], dtype=float)

print("Imagem original:")
print(image)

# Define parâmetros
kernel_size = 3
iterations = 2

# Aplica o filtro
blurred_image = iterative_gaussian_blur(image, kernel_size, iterations)

print("\nImagem após filtro (arredondada):")
print(np.round(blurred_image, 2))

## Observações finais

Compare os resultados da versão em C com esta implementação sequencial em Python. Pequenas diferenças podem ocorrer devido à precisão de ponto flutuante.